In [8]:
import pandas as pd
import numpy as np

file1 = pd.read_csv("bestSelling_games.csv")
file2 = pd.read_csv("steam_games_2026.csv")

file1["game_name"] = file1["game_name"].str.lower().str.strip()
file2["Name"] = file2["Name"].str.lower().str.strip()

df = pd.merge(file1, file2, left_on="game_name", right_on="Name", how="inner")

print(df.shape)
df.head()

(424, 27)


,game_name,reviews_like_rate,all_reviews_number,release_date,developer,user_defined_tags,supported_os,supported_languages,price,other_features,...,Release_Date,Primary_Genre,All_Tags,Price_USD,Discount_Pct,Review_Score_Pct,Total_Reviews,Steam_Deck_Status,Estimated_Owners,24h_Peak_Players
0,counter-strike 2,86,8803754,"21 Aug, 2012",Valve,"FPS, Action, Tactical","win, linux","English, Czech, Danish, Dutch, Finnish, French...",0.00,"Cross-Platform Multiplayer, Steam Trading Card...",...,2012-08-21,Action,FPS;Shooter;Multiplayer;Competitive;Action;Tea...,0.00,0,83,4980365,Unknown,149410950,1013936
1,pubg: battlegrounds,59,2554482,"21 Dec, 2017",PUBG Corporation,"Survival, Shooter, Action, Tactical",win,"English, Korean, Simplified Chinese, French, G...",0.00,"Online PvP, Stats, Remote Play on Phone, Remot...",...,2017-12-21,Action,Survival;Shooter;Battle Royale;Multiplayer;FPS...,0.00,0,74,1757549,Unknown,52726470,314682
2,elden ring nightreign,77,53426,"30 May, 2025","FromSoftware, Inc.","Souls-like, Open World, Fantasy, RPG",win,"English, Japanese, French, Italian, German, Sp...",25.99,"Single-player, Online Co-op, Steam Achievement...",...,2025-05-29,Action,Souls-like;Online Co-Op;Multiplayer;Roguelike;...,39.99,0,80,122006,Unknown,3660180,163599
3,the last of us™ part i,79,45424,"28 Mar, 2023",Naughty Dog LLC,"Story Rich, Shooter, Survival, Horror",win,"English, Italian, Spanish - Spain, Czech, Dutc...",59.99,"Single-player, Steam Achievements, Steam Tradi...",...,2023-03-28,Action,Story Rich;Post-apocalyptic;Zombies;Horror;Act...,59.99,0,95,59411,Unknown,1782330,4144
4,red dead redemption 2,92,672140,"5 Dec, 2019",Rockstar Games,"Open World, Story Rich, Adventure, Realistic, ...",win,"English, French, Italian, German, Spanish - Sp...",59.99,"Single-player, Online PvP, Online Co-op, Steam...",...,2019-12-05,Action,Open World;Story Rich;Western;Multiplayer;Adve...,59.99,0,95,785860,Unknown,23575800,29885


In [9]:
import numpy as np

X = df[["price", "Review_Score_Pct"]]

y = np.log1p(df["estimated_downloads"])

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [12]:
score = model.score(X_test, y_test)
print("R² score:", score)

R² score: 0.006815075715076335


A linear regression model was used to predict log-transformed downloads based on price and review score.

The model achieved an R² score of 0.007, indicating that these features explain almost none of the variation in downloads. This suggests that price and review score alone are not sufficient to predict game popularity.

Game popularity is likely influenced by more complex factors such as genre, marketing, player trends, and community engagement.

This result is consistent with earlier exploratory data analysis, where no strong relationships were observed between these variables and downloads.

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

df["popular"] = (df["estimated_downloads"] > df["estimated_downloads"].median()).astype(int)

X = df[["price", "Review_Score_Pct"]]

y = df["popular"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

y_pred = tree_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Accuracy: 0.4470588235294118
              precision    recall  f1-score   support

           0       0.45      0.55      0.49        42
           1       0.44      0.35      0.39        43

    accuracy                           0.45        85
   macro avg       0.45      0.45      0.44        85
weighted avg       0.45      0.45      0.44        85



A decision tree classifier was used to classify games as popular or not popular. Games with estimated downloads above the median were labeled as popular, while the rest were labeled as not popular.

The model used price and review score as input features. The accuracy score shows how well the model predicted the popularity category on unseen test data.

This classification task complements the regression model by approaching popularity as a category instead of a continuous numerical value.

The decision tree classifier achieved an accuracy of approximately 0.447. Since this is close to random guessing for a binary classification problem, the result suggests that price and review score alone are not strong enough features to classify games as popular or not popular. This supports the regression result, where price and review score also explained very little of the variation in downloads.